# Python OOP Refresher

**Module:** 1 — Foundations & Data Engineering  
**Week:** 1, Day 1  

## Learning Objectives
- Understand classes, instances, and the `self` parameter
- Use `__init__`, `__repr__`, `__str__`, and other dunder methods
- Implement inheritance and method overriding
- Use `@property`, `@classmethod`, `@staticmethod`
- Work with dataclasses for cleaner data containers
- Apply type hints for better code clarity

## Resources
- [Corey Schafer: Python OOP Playlist](https://www.youtube.com/playlist?list=PL-osiE80TeTsqhIuOqKhwlXsIBIdSeYtc)
- [Real Python: Python Classes](https://realpython.com/python3-object-oriented-programming/)
- [Python docs: Classes](https://docs.python.org/3/tutorial/classes.html)

---
## 1. Classes & Instances

A **class** is a blueprint for creating objects. An **instance** is a specific object created from that class.

Think of a class like a cookie cutter and instances like the cookies.

In [ ]:
class Product:
    """A simple product in an e-commerce store."""
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        self.name = name
        self.price = price
        self.quantity = quantity
    
    def total_value(self) -> float:
        """Calculate total inventory value for this product."""
        return self.price * self.quantity


# Create instances
laptop = Product("Laptop", 999.99, 50)
mouse = Product("Mouse", 29.99, 200)

print(f"{laptop.name}: ${laptop.total_value():,.2f} in stock")
print(f"{mouse.name}: ${mouse.total_value():,.2f} in stock")

### Key Points
- `__init__` is the **constructor** — called when you create a new instance
- `self` refers to the specific instance being created/used
- Type hints (`: str`, `: float`, `-> float`) don't enforce types but document intent

---
## 2. Dunder (Magic) Methods

Special methods surrounded by double underscores that customize class behavior.

In [ ]:
class Product:
    """Product with rich comparison and string representations."""
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        self.name = name
        self.price = price
        self.quantity = quantity
    
    def __repr__(self) -> str:
        """Developer-friendly representation (for debugging)."""
        return f"Product('{self.name}', {self.price}, {self.quantity})"
    
    def __str__(self) -> str:
        """User-friendly representation (for display)."""
        return f"{self.name} — ${self.price:.2f} ({self.quantity} in stock)"
    
    def __eq__(self, other) -> bool:
        """Two products are equal if they have the same name."""
        if not isinstance(other, Product):
            return NotImplemented
        return self.name == other.name
    
    def __lt__(self, other) -> bool:
        """Compare products by price."""
        if not isinstance(other, Product):
            return NotImplemented
        return self.price < other.price
    
    def __len__(self) -> int:
        """Length = quantity in stock."""
        return self.quantity


laptop = Product("Laptop", 999.99, 50)
mouse = Product("Mouse", 29.99, 200)

# __repr__ vs __str__
print(repr(laptop))   # Product('Laptop', 999.99, 50)
print(str(laptop))    # Laptop — $999.99 (50 in stock)
print()               

# __eq__
laptop2 = Product("Laptop", 1099.99, 10)
print(f"laptop == laptop2? {laptop == laptop2}")  # True (same name)
print(f"laptop == mouse? {laptop == mouse}")      # False
print()

# __lt__ enables sorting
products = [laptop, mouse, Product("Keyboard", 79.99, 100)]
print("Sorted by price:", [p.name for p in sorted(products)])
print()

# __len__
print(f"len(mouse) = {len(mouse)}")

### Common Dunder Methods

| Method | Purpose | Example |
|--------|---------|--------|
| `__init__` | Constructor | `obj = MyClass()` |
| `__repr__` | Debug string | `repr(obj)` |
| `__str__` | Display string | `str(obj)`, `print(obj)` |
| `__eq__` | Equality | `obj1 == obj2` |
| `__lt__` | Less than | `obj1 < obj2`, `sorted()` |
| `__len__` | Length | `len(obj)` |
| `__getitem__` | Indexing | `obj[key]` |
| `__contains__` | Membership | `item in obj` |
| `__add__` | Addition | `obj1 + obj2` |

---
## 3. Inheritance & Method Overriding

A child class **inherits** attributes and methods from a parent class, and can **override** them.

In [ ]:
class Product:
    """Base product class."""
    tax_rate = 0.08  # Class variable — shared by all instances
    
    def __init__(self, name: str, price: float, quantity: int = 0):
        self.name = name
        self.price = price
        self.quantity = quantity
    
    def total_price(self) -> float:
        """Price including tax."""
        return self.price * (1 + self.tax_rate)
    
    def __repr__(self) -> str:
        return f"{self.__class__.__name__}('{self.name}', {self.price})"


class DigitalProduct(Product):
    """Digital products have no tax in many jurisdictions."""
    tax_rate = 0.0  # Override class variable
    
    def __init__(self, name: str, price: float, download_url: str):
        super().__init__(name, price, quantity=999)  # Unlimited stock
        self.download_url = download_url


class PerishableProduct(Product):
    """Products with an expiry date."""
    
    def __init__(self, name: str, price: float, quantity: int, expiry_date: str):
        super().__init__(name, price, quantity)
        self.expiry_date = expiry_date
    
    def total_price(self) -> float:
        """Override: 50% discount if near expiry (simplified)."""
        base = super().total_price()
        # In real code, you'd check if expiry_date is within N days
        return base * 0.5  # Simulating near-expiry discount


# Usage
laptop = Product("Laptop", 1000)
ebook = DigitalProduct("Python Guide", 29.99, "https://example.com/book.pdf")
milk = PerishableProduct("Milk", 4.99, 100, "2025-04-01")

for item in [laptop, ebook, milk]:
    print(f"{item!r:45s} → total: ${item.total_price():.2f}")

# isinstance checks
print(f"\nIs ebook a Product? {isinstance(ebook, Product)}")
print(f"Is laptop a DigitalProduct? {isinstance(laptop, DigitalProduct)}")

### Key Points
- `super().__init__()` calls the parent's constructor
- Child classes can override methods and class variables
- `isinstance()` checks if an object is an instance of a class (including parents)
- **Class variables** are shared; **instance variables** (set via `self.x`) are per-instance

---
## 4. Properties, Class Methods & Static Methods

In [ ]:
class BankAccount:
    """Demonstrates @property, @classmethod, @staticmethod."""
    
    _interest_rate = 0.02  # "Private" class variable (convention)
    
    def __init__(self, owner: str, balance: float = 0):
        self.owner = owner
        self._balance = balance  # "Private" — use property to access
    
    # ---- @property: controlled access to attributes ----
    @property
    def balance(self) -> float:
        """Read-only access to balance."""
        return self._balance
    
    @property
    def balance_with_interest(self) -> float:
        """Computed property — balance after interest."""
        return self._balance * (1 + self._interest_rate)
    
    # ---- Regular methods: operate on the instance ----
    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount
    
    def withdraw(self, amount: float) -> None:
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
    
    # ---- @classmethod: operates on the class, not instances ----
    @classmethod
    def set_interest_rate(cls, rate: float) -> None:
        """Change interest rate for ALL accounts."""
        cls._interest_rate = rate
    
    @classmethod
    def from_string(cls, account_str: str) -> "BankAccount":
        """Alternative constructor: 'Alice:5000' → BankAccount('Alice', 5000)."""
        owner, balance = account_str.split(":")
        return cls(owner, float(balance))
    
    # ---- @staticmethod: utility, no access to cls or self ----
    @staticmethod
    def validate_amount(amount: float) -> bool:
        """Check if an amount is valid."""
        return isinstance(amount, (int, float)) and amount > 0
    
    def __repr__(self):
        return f"BankAccount('{self.owner}', {self._balance:.2f})"


# @property
acc = BankAccount("Alice", 1000)
print(f"Balance: ${acc.balance:.2f}")                    # Uses @property getter
print(f"With interest: ${acc.balance_with_interest:.2f}") # Computed property
# acc.balance = 5000  # ← Would raise AttributeError (read-only)

# Regular methods
acc.deposit(500)
print(f"After deposit: ${acc.balance:.2f}")

# @classmethod — alternative constructor
bob = BankAccount.from_string("Bob:3000")
print(f"\nBob's account: {bob}")

# @staticmethod — utility
print(f"\nIs 100 valid? {BankAccount.validate_amount(100)}")
print(f"Is -50 valid? {BankAccount.validate_amount(-50)}")

### When to Use What

| Decorator | Has access to | Use case |
|-----------|--------------|----------|
| *(none)* — regular method | `self` (instance) | Most methods — operate on instance data |
| `@property` | `self` (instance) | Controlled attribute access, computed values |
| `@classmethod` | `cls` (class) | Alternative constructors, modifying class state |
| `@staticmethod` | *(nothing)* | Utility functions logically grouped with the class |

---
## 5. Dataclasses

Python 3.7+ provides `@dataclass` to automatically generate `__init__`, `__repr__`, `__eq__`, and more.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Customer:
    """A customer with automatic __init__, __repr__, __eq__."""
    name: str
    email: str
    purchases: list[float] = field(default_factory=list)
    
    @property
    def total_spent(self) -> float:
        return sum(self.purchases)
    
    @property
    def average_purchase(self) -> float:
        if not self.purchases:
            return 0.0
        return self.total_spent / len(self.purchases)
    
    def add_purchase(self, amount: float) -> None:
        self.purchases.append(amount)


# No need to write __init__ or __repr__!
alice = Customer("Alice", "alice@example.com", [99.99, 49.99, 199.99])
bob = Customer("Bob", "bob@example.com")

print(alice)  # Auto-generated __repr__
print(f"Alice's total: ${alice.total_spent:.2f}")
print(f"Alice's average: ${alice.average_purchase:.2f}")
print()

bob.add_purchase(29.99)
print(f"Bob: {bob}")

# Auto-generated __eq__ compares all fields
alice2 = Customer("Alice", "alice@example.com", [99.99, 49.99, 199.99])
print(f"\nalice == alice2? {alice == alice2}")

### Dataclass Tips
- Use `field(default_factory=list)` for mutable defaults (never `purchases: list = []`!)
- Add `frozen=True` to make instances immutable: `@dataclass(frozen=True)`
- Add `order=True` to auto-generate comparison methods
- You can still add regular methods and `@property`

---
## 6. Putting It Together: A Mini Data Pipeline Class

This pattern will be useful throughout the curriculum — encapsulating a multi-step process in a class.

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class PipelineStep:
    """A single step in a data pipeline."""
    name: str
    func: Any  # callable
    description: str = ""


class DataPipeline:
    """A simple sequential data pipeline."""
    
    def __init__(self, name: str):
        self.name = name
        self._steps: list[PipelineStep] = []
        self._log: list[str] = []
    
    def add_step(self, name: str, func, description: str = "") -> "DataPipeline":
        """Add a step. Returns self for chaining."""
        self._steps.append(PipelineStep(name, func, description))
        return self  # enables method chaining
    
    def run(self, data):
        """Execute all steps sequentially."""
        self._log = []
        result = data
        
        for i, step in enumerate(self._steps, 1):
            self._log.append(f"Step {i}/{len(self._steps)}: {step.name}")
            result = step.func(result)
            self._log.append(f"  ✓ Complete — output shape: {getattr(result, 'shape', 'N/A')}")
        
        return result
    
    def print_log(self):
        """Print execution log."""
        print(f"\n--- Pipeline: {self.name} ---")
        for entry in self._log:
            print(entry)
    
    def __repr__(self):
        step_names = [s.name for s in self._steps]
        return f"DataPipeline('{self.name}', steps={step_names})"
    
    def __len__(self):
        return len(self._steps)

In [ ]:
import pandas as pd
import numpy as np

# Create sample data
np.random.seed(42)
raw_data = pd.DataFrame({
    "name": ["Alice", "Bob", None, "Diana", "Eve", "Frank"],
    "age": [25, 30, 35, None, 28, 45],
    "salary": [50000, 60000, 75000, 80000, None, 90000],
    "department": ["Engineering", "Marketing", "Engineering", "Sales", "Marketing", "Engineering"]
})

print("Raw data:")
print(raw_data)
print(f"\nMissing values:\n{raw_data.isnull().sum()}")

In [ ]:
# Define transformation functions
def drop_missing_names(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna(subset=["name"])

def fill_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["age"] = df["age"].fillna(df["age"].median())
    df["salary"] = df["salary"].fillna(df["salary"].median())
    return df

def add_salary_band(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["salary_band"] = pd.cut(df["salary"], bins=[0, 60000, 80000, float("inf")],
                                labels=["Junior", "Mid", "Senior"])
    return df

# Build and run the pipeline using method chaining
pipeline = (
    DataPipeline("Employee Cleanup")
    .add_step("Drop missing names", drop_missing_names)
    .add_step("Fill missing values", fill_missing_values)
    .add_step("Add salary bands", add_salary_band)
)

print(f"Pipeline: {pipeline}")
print(f"Steps: {len(pipeline)}")

clean_data = pipeline.run(raw_data)
pipeline.print_log()

print("\nClean data:")
print(clean_data)

---
## 7. Practice Exercises

Try these on your own before looking at solutions.

### Exercise 1: Shopping Cart
Create a `ShoppingCart` class with:
- An `add_item(product, quantity)` method
- A `remove_item(product)` method
- A `total` property that computes the total price
- `__len__` returns the number of unique items
- `__contains__` checks if a product is in the cart

In [ ]:
# Exercise 1: Your code here
class ShoppingCart:
    pass


# Test it:
# cart = ShoppingCart()
# cart.add_item(Product("Laptop", 999.99), 1)
# cart.add_item(Product("Mouse", 29.99), 2)
# print(f"Total: ${cart.total:.2f}")
# print(f"Items: {len(cart)}")

### Exercise 2: Employee Hierarchy
Create a class hierarchy:
- `Employee` (base): name, salary, `annual_cost()` method
- `Manager(Employee)`: has a team (list of Employees), `annual_cost()` includes team costs
- `Intern(Employee)`: fixed 3-month salary, override `annual_cost()`

Use `@dataclass` for at least one of these.

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Extend the Pipeline
Add error handling to `DataPipeline`:
- If a step raises an exception, log the error and stop execution
- Add a `dry_run()` method that prints the steps without executing them
- Add a `remove_step(name)` method

In [ ]:
# Exercise 3: Your code here


---
## Key Takeaways

1. **Classes** encapsulate data + behavior. Use them when you have related data and operations.
2. **Dunder methods** make your classes work with Python builtins (`print`, `len`, `sorted`, `==`).
3. **Inheritance** lets you specialize behavior. Use `super()` to call parent methods.
4. **`@property`** gives controlled access to attributes — use it instead of getters/setters.
5. **`@dataclass`** eliminates boilerplate for data-holding classes.
6. **Method chaining** (returning `self`) makes APIs fluent and readable.

### When to Use OOP vs Functions
- **Use classes** when you have state that persists across operations (pipelines, models, connections)
- **Use functions** for stateless transformations (pure data processing)
- **In data science**: Classes are great for pipelines, custom transformers, and API wrappers. Functions are great for individual data transformations.